# EduPredict: Student Academic Performance Prediction Using Artificial Neural Networks
### Master of Business Administration (MBA) – Data Science & Artificial Intelligence
---
**Author:** Data Science & AI Research Team  
**Objective:** Build, train, evaluate, and interpret a Multilayer Perceptron (ANN) to predict student success/failure from multi-attribute behavioral and academic factors.

### Pipeline Overview:
1. **Synthetic Cohort Generation** (Deterministic, multi-feature with non-linear correlations)
2. **Exploratory Data Analysis (EDA)** & Statistical Distributions
3. **Leakage-Free Preprocessing** (StandardScaler + One-Hot Encoding)
4. **Artificial Neural Network Architecture** (Feedforward MLP with ReLU, Dropout, Sigmoid)
5. **Model Training & Convergence** (Adam Optimizer, Binary Cross-Entropy)
6. **Evaluation & Diagnostic Curves** (Confusion Matrix, Precision-Recall, ROC-AUC)
7. **Weight Serialization & Artifact Export** for TensorFlow.js Dashboard

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score

%matplotlib inline
sns.set_theme(style="whitegrid", palette="deep")
print("Libraries successfully imported!")

## 1. Synthetic Student Cohort Generation
In educational data mining, realistic features include attendance rate, daily study hours, historical exam performance, assignment completion, classroom participation, sleep regularity, and home internet connectivity.

In [ ]:
def generate_student_cohort(n=2000, seed=42, passing_score=40.0):
    rng = np.random.default_rng(seed)
    student_id = [f"STU{str(i+1).zfill(4)}" for i in range(n)]
    age = rng.integers(16, 26, n)
    gender = rng.choice(["Male", "Female"], n, p=[0.52, 0.48])
    attendance = rng.uniform(40, 100, n).round(2)
    study_hours = rng.uniform(0, 10, n).round(2)
    prev_score = rng.uniform(20, 100, n).round(2)
    assign_score = rng.uniform(20, 100, n).round(2)
    participation = rng.uniform(0, 100, n).round(2)
    sleep_hours = rng.uniform(3, 10, n).round(2)
    internet = rng.choice(["Yes", "No"], n, p=[0.75, 0.25])
    extra = rng.choice(["Yes", "No"], n, p=[0.45, 0.55])

    # Non-linear composite score with stochastic variance
    internet_boost = np.where(internet == "Yes", 5.0, 0.0)
    extra_effect = np.where(extra == "Yes", 3.0, -2.0)
    sleep_effect = np.clip((sleep_hours - 6.0) * 2.0, -6.0, 6.0)

    raw_final = (
        0.30 * attendance
        + 0.25 * study_hours * 10
        + 0.25 * prev_score
        + 0.10 * assign_score
        + 0.05 * participation
        + internet_boost
        + extra_effect
        + sleep_effect
        + rng.normal(0, 6, n)
    )

    final_score = np.clip(raw_final, 0, 100).round(2)
    performance = np.where(final_score >= passing_score, "Pass", "Fail")

    return pd.DataFrame({
        "Student_ID": student_id,
        "Age": age,
        "Gender": gender,
        "Attendance": attendance,
        "Study_Hours": study_hours,
        "Previous_Score": prev_score,
        "Assignment_Score": assign_score,
        "Participation": participation,
        "Sleep_Hours": sleep_hours,
        "Internet_Access": internet,
        "Extracurricular": extra,
        "Final_Score": final_score,
        "Performance": performance
    })

df = generate_student_cohort(n=2000)
print(f"Cohort shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print("Class Distribution:")
print(df["Performance"].value_counts(normalize=True).round(4) * 100)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["Attendance"], kde=True, ax=axes[0], color="royalblue").set(title="Attendance Distribution")
sns.histplot(df["Study_Hours"], kde=True, ax=axes[1], color="seagreen").set(title="Daily Study Hours")
sns.boxplot(x="Performance", y="Final_Score", data=df, ax=axes[2], palette=["mediumseagreen", "crimson"]).set(title="Final Score by Class")
plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Target Isolation
To prevent data leakage, transformers must be fit solely on the training fold.

In [ ]:
num_cols = ["Age", "Attendance", "Study_Hours", "Previous_Score", "Assignment_Score", "Participation", "Sleep_Hours"]
cat_cols = ["Gender", "Internet_Access", "Extracurricular"]

# Target encoding: Pass = 1, Fail = 0
y = (df["Performance"] == "Pass").astype(int).values

# Split before scaling (80/20)
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.20, random_state=42, stratify=y)

# Fit scaler on train
train_num = df.iloc[train_idx][num_cols].values
means = train_num.mean(axis=0)
stds = train_num.std(axis=0)
stds[stds == 0] = 1.0

# Transform train & test
X_num_train = (train_num - means) / stds
X_num_test = (df.iloc[test_idx][num_cols].values - means) / stds

# One-Hot Categoricals
X_cat_train = pd.get_dummies(df.iloc[train_idx][cat_cols]).astype(float).values
X_cat_test = pd.get_dummies(df.iloc[test_idx][cat_cols]).astype(float).values

X_train = np.hstack([X_num_train, X_cat_train])
X_test = np.hstack([X_num_test, X_cat_test])
y_train = y[train_idx]
y_test = y[test_idx]

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 4. Artificial Neural Network Training (MLPClassifier)
Configuring a 2-hidden layer Feedforward Neural Network: `Input(13) -> Dense(64, ReLU) -> Dense(32, ReLU) -> Output(1, Sigmoid)`.

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=200,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15,
    verbose=False
)

mlp.fit(X_train, y_train)
print(f"Model converged in {mlp.n_iter_} iterations with final loss {mlp.loss_:.4f}")

## 5. Model Evaluation on Holdout Test Set

In [ ]:
y_proba = mlp.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.50).astype(int)

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=["Fail", "Pass"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0], xticklabels=["Pred Fail", "Pred Pass"], yticklabels=["Actual Fail", "Actual Pass"])
ax[0].set_title("Test Confusion Matrix")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC = {roc_auc_score(y_test, y_proba):.3f})")
ax[1].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
ax[1].set_xlabel("False Positive Rate")
ax[1].set_ylabel("True Positive Rate")
ax[1].set_title("ROC Curve")
ax[1].legend(loc="lower right")
plt.show()

## 6. Real-Time Student Prediction Demo

In [ ]:
# Example new student profile
new_student = {
    "Age": 21,
    "Attendance": 82.5,
    "Study_Hours": 6.0,
    "Previous_Score": 72.0,
    "Assignment_Score": 85.0,
    "Participation": 65.0,
    "Sleep_Hours": 7.5,
    "Gender_Female": 1.0,
    "Gender_Male": 0.0,
    "Internet_Access_No": 0.0,
    "Internet_Access_Yes": 1.0,
    "Extracurricular_No": 0.0,
    "Extracurricular_Yes": 1.0
}

num_features = np.array([[new_student[c] for c in num_cols]])
scaled_num = (num_features - means) / stds
cat_features = np.array([[new_student["Gender_Female"], new_student["Gender_Male"], new_student["Internet_Access_No"], new_student["Internet_Access_Yes"], new_student["Extracurricular_No"], new_student["Extracurricular_Yes"]]])
input_vector = np.hstack([scaled_num, cat_features])

prob_pass = mlp.predict_proba(input_vector)[0, 1]
print(f"Predicted Probability of Passing: {prob_pass*100:.2f}%")
print(f"Verdict: {'PASS' if prob_pass >= 0.50 else 'FAIL / AT RISK'}")